In [1]:
### heuristictesting.ipynb is a modular and agnostic notebook for testing the M heuristic on different
### horizons.

import pandas as pd
import numpy as np

def m_heuristic_accuracy(df, margin_col='margin', price_open_col='value', price_close_col='value',
                         rows_per_M=12, M_start=1, M_end=10, M_step=0.25):
    """
    Calculate heuristic accuracy across horizons.
    
    Args:
        df : DataFrame with columns for margin, open, close.
        margin_col : str, column name for margin (B - S).
        price_open_col : str, column for horizon open price.
        price_close_col : str, column for horizon close price.
        rows_per_M : int, number of rows per M1 (12 for S5 rows, 1 for M1Q0 backtest output).
        M_start, M_end, M_step : horizon range.

    Returns:
        DataFrame per horizon M with:
        Acc, Acc Price Up, Acc Price Down : margin sign vs price change in the SAME block (descriptive).
        Acc Next : margin sign vs price change in the NEXT block (predictive, what trading needs).
        Baseline Next : hit rate of always guessing the majority next-block direction. Acc Next
                        must beat this, not just 0.5, to carry information.
        Pred Up : share of blocks where the margin says up. Near 1 or 0 means it is not discriminating.
    """
    results = []
    
    def horizon_accuracy(block):
        margin_block = block[margin_col].mean()
        price_change = block[price_close_col].iloc[-1] - block[price_open_col].iloc[0]
        return np.sign(margin_block), np.sign(price_change)

    for m in np.arange(M_start, M_end + M_step, M_step):
        block_size = int(rows_per_M * m)
        
        total_blocks = total_right = 0
        total_p = total_n = 0
        right_p = right_n = 0
        m_signs, p_signs = [], []

        for i in range(0, len(df), block_size):
            block = df.iloc[i:i+block_size]
            if len(block) < block_size:
                continue
            m_sign, p_sign = horizon_accuracy(block)
            m_signs.append(m_sign)
            p_signs.append(p_sign)
            
            total_blocks += 1

            if m_sign == p_sign:
                total_right += 1

                if p_sign > 0:
                    total_p += 1
                    right_p += 1
                elif p_sign < 0:
                    total_n += 1
                    right_n += 1
            else:
                if p_sign > 0:
                    total_p += 1
                elif p_sign < 0:
                    total_n += 1

        
        acc = total_right / total_blocks
        acc_p = right_p / total_p if total_p else np.nan
        acc_n = right_n / total_n if total_n else np.nan

        ## Predictive: this block's margin against the next block's price change, flat blocks skipped
        m_now, p_next = np.array(m_signs[:-1]), np.array(p_signs[1:])
        keep = (m_now != 0) & (p_next != 0)
        acc_next = (m_now[keep] == p_next[keep]).mean() if keep.any() else np.nan
        up_next = (p_next[keep] > 0).mean() if keep.any() else np.nan
        baseline_next = max(up_next, 1 - up_next)
        pred_up = (m_now[keep] > 0).mean() if keep.any() else np.nan

        results.append((round(m,2), block_size,total_blocks ,  acc , acc_p , acc_n , acc_next , baseline_next , pred_up))
    
    return pd.DataFrame(results, columns=['M','RpH', 'Hcount','Acc' , 'Acc Price Up' , 'Acc Price Down' ,
                                          'Acc Next' , 'Baseline Next' , 'Pred Up'])

In [29]:
## 9/29 to 10/3 , tuned for this week... -> frozen

trainingweek = pd.read_csv("full_labeled_USD_JPY_2025-09-29_to_2025-10-03.csv")
accuracyontraining = m_heuristic_accuracy(trainingweek , M_end=30) # up to 30 minutes
print(m_heuristic_accuracy(trainingweek , M_end = 30))

         M  RpH  Hcount       Acc  Acc Price Up  Acc Price Down
0     1.00   12    6451  0.472640      0.102887        0.892473
1     1.25   15    5160  0.487016      0.081769        0.925689
2     1.50   18    4300  0.496279      0.060666        0.942389
3     1.75   21    3686  0.495388      0.047258        0.958791
4     2.00   24    3225  0.491163      0.035503        0.965300
..     ...  ...     ...       ...           ...             ...
112  29.00  348     222  0.509009      0.000000        1.000000
113  29.25  351     220  0.563636      0.000000        1.000000
114  29.50  354     218  0.573394      0.000000        1.000000
115  29.75  357     216  0.560185      0.000000        1.000000
116  30.00  360     215  0.595349      0.000000        1.000000

[117 rows x 6 columns]


In [30]:
## Median Accuracy of Heuristic from M1 to M30 and send to .csv
print(f"The median accuracy across horizons from M1 to M30 is {np.median(accuracyontraining['Acc'])}")
accuracyontraining.to_csv("accuracyoverseptember29tooct03M1toM30.csv" , index = False)

The median accuracy across horizons from M1 to M30 is 0.528052805280528


In [ ]:
## 9/22 to 09/26 , tuned for this week... -> frozen

trainingweek = pd.read_csv("full_labeled_USD_JPY_2025-09-22_to_2025-09-26.csv")
accuracyontraining = m_heuristic_accuracy(trainingweek , M_end=30) # up to 30 minutes
print(m_heuristic_accuracy(trainingweek , M_end = 30))

         M  RpH  Hcount       Acc  Acc Price Up  Acc Price Down
0     1.00   12    6450  0.485271      0.951503        0.063584
1     1.25   15    5160  0.484109      0.973071        0.032272
2     1.50   18    4300  0.484884      0.978846        0.024873
3     1.75   21    3685  0.487381      0.990503        0.013380
4     2.00   24    3225  0.499535      0.990099        0.007488
..     ...  ...     ...       ...           ...             ...
112  29.00  348     222  0.518018      1.000000        0.000000
113  29.25  351     220  0.504545      1.000000        0.000000
114  29.50  354     218  0.481651      1.000000        0.000000
115  29.75  357     216  0.523148      1.000000        0.000000
116  30.00  360     215  0.511628      1.000000        0.000000

[117 rows x 6 columns]


In [40]:
## Median Accuracy of Heuristic from M1 to M30 and send to .csv
print(f"The median accuracy across horizons from M1 to M30 is {np.median(accuracyontraining['Acc'])}")
accuracyontraining.to_csv("accuracyoverseptember22tosept26M1toM30.csv" , index = False)

The median accuracy across horizons from M1 to M30 is 0.5065524193548387


In [ ]:
## 9/15 to 09/19 , tuned for this week... -> frozen

trainingweek = pd.read_csv("full_labeled_USD_JPY_2025-09-15_to_2025-09-19.csv")
accuracyontraining = m_heuristic_accuracy(trainingweek , M_end=30) # up to 30 minutes
print(m_heuristic_accuracy(trainingweek , M_end = 30))

         M  RpH  Hcount       Acc  Acc Price Up  Acc Price Down
0     1.00   12    6511  0.478728      0.876123        0.126885
1     1.25   15    5208  0.487903      0.908127        0.094059
2     1.50   18    4340  0.483871      0.927515        0.075097
3     1.75   21    3720  0.486559      0.948419        0.056561
4     2.00   24    3255  0.489401      0.960352        0.043338
..     ...  ...     ...       ...           ...             ...
112  29.00  348     224  0.540179      1.000000        0.000000
113  29.25  351     222  0.477477      1.000000        0.000000
114  29.50  354     220  0.531818      1.000000        0.000000
115  29.75  357     218  0.522936      1.000000        0.000000
116  30.00  360     217  0.520737      1.000000        0.000000

[117 rows x 6 columns]


In [47]:
## Median Accuracy of Heuristic from M1 to M30 and send to .csv
print(f"The median accuracy across horizons from M1 to M30 is {np.median(accuracyontraining['Acc'])}")
accuracyontraining.to_csv("accuracyoverseptember15tosept19M1toM30.csv" , index = False)

The median accuracy across horizons from M1 to M30 is 0.4921135646687697


In [ ]:
## 9/08 to 09/12 , tuned for this week... -> frozen

trainingweek = pd.read_csv("full_labeled_USD_JPY_2025-09-08_to_2025-09-12.csv")
accuracyontraining = m_heuristic_accuracy(trainingweek , M_end=30) # up to 30 minutes
print(m_heuristic_accuracy(trainingweek , M_end = 30))

         M  RpH  Hcount       Acc  Acc Price Up  Acc Price Down
0     1.00   12    6401  0.466021      0.931712        0.061164
1     1.25   15    5121  0.466120      0.950062        0.042276
2     1.50   18    4267  0.482540      0.974720        0.026812
3     1.75   21    3658  0.469656      0.978674        0.011142
4     2.00   24    3200  0.477812      0.990164        0.012110
..     ...  ...     ...       ...           ...             ...
112  29.00  348     220  0.568182      1.000000        0.000000
113  29.25  351     218  0.550459      1.000000        0.000000
114  29.50  354     217  0.511521      1.000000        0.000000
115  29.75  357     215  0.544186      1.000000        0.000000
116  30.00  360     213  0.521127      1.000000        0.000000

[117 rows x 6 columns]


In [51]:
## Median Accuracy of Heuristic from M1 to M30 and send to .csv
print(f"The median accuracy across horizons from M1 to M30 is {np.median(accuracyontraining['Acc'])}")
accuracyontraining.to_csv("accuracyoverseptember08tosept12M1toM30.csv" , index = False)

The median accuracy across horizons from M1 to M30 is 0.5008201202843083


In [4]:
## 10/06 to 10/10 , tuned for this week -> discretionary use of MREG = 1 because uniform signal

trainingweek = pd.read_csv("full_labeled_USD_JPY_2025-10-06_to_2025-10-10.csv")
accuracyontraining = m_heuristic_accuracy(trainingweek , M_end=30) # up to 30 minutes
print(m_heuristic_accuracy(trainingweek , M_end = 30))

         M  RpH  Hcount       Acc  Acc Price Up  Acc Price Down
0     1.00   12    6544  0.482427      0.895002        0.101506
1     1.25   15    5235  0.483477      0.918367        0.077864
2     1.50   18    4363  0.488196      0.933796        0.055176
3     1.75   21    3739  0.488633      0.955386        0.039933
4     2.00   24    3272  0.504279      0.974468        0.030679
..     ...  ...     ...       ...           ...             ...
112  29.00  348     225  0.533333      1.000000        0.000000
113  29.25  351     223  0.520179      1.000000        0.000000
114  29.50  354     221  0.574661      1.000000        0.000000
115  29.75  357     219  0.538813      1.000000        0.000000
116  30.00  360     218  0.541284      1.000000        0.000000

[117 rows x 6 columns]


In [5]:
## Median Accuracy of Heuristic from M1 to M30 and send to .csv
print(f"The median accuracy across horizons from M1 to M30 is {np.median(accuracyontraining['Acc'])}")
accuracyontraining.to_csv("accuracyoveroctober06tooctober10M1toM30.csv" , index = False)

The median accuracy across horizons from M1 to M30 is 0.5201793721973094


In [4]:
## 03 2026 signal testing for heuristic for accuracy across multiple different granularities

trainingweek = pd.read_csv("full_labeled_USD_JPY_2026-03-01_to_2026-03-31.csv")
accuracyontraining = m_heuristic_accuracy(trainingweek , M_end=30) # up to 30 minutes
print(m_heuristic_accuracy(trainingweek , M_end = 30))

         M  RpH  Hcount       Acc  Acc Price Up  Acc Price Down
0     1.00   12   29744  0.482988      0.879317        0.117965
1     1.25   15   23795  0.484261      0.903429        0.092368
2     1.50   18   19829  0.485904      0.925067        0.066667
3     1.75   21   16996  0.490527      0.941876        0.051292
4     2.00   24   14872  0.492200      0.956916        0.036449
..     ...  ...     ...       ...           ...             ...
112  29.00  348    1025  0.518049      1.000000        0.000000
113  29.25  351    1016  0.523622      1.000000        0.000000
114  29.50  354    1008  0.514881      1.000000        0.000000
115  29.75  357     999  0.526527      1.000000        0.000000
116  30.00  360     991  0.535822      1.000000        0.000000

[117 rows x 6 columns]


In [5]:
## Median Accuracy of Heuristic from M1 to M30 and send to .csv
print(f"The median accuracy across horizons from M1 to M30 is {np.median(accuracyontraining['Acc'])}")
accuracyontraining.to_csv("accuracyovermarch2026M1toM30.csv" , index = False)

The median accuracy across horizons from M1 to M30 is 0.5205831903945112
